# P0 · Benchmark bộ sinh công thức trên Kaggle

Chạy 4 ứng viên (`bench/models.yaml`) trên 76 prompt × 3 seed, lượng tử NF4, 2 card T4 chạy song song.

**Cài đặt notebook trước khi chạy** (panel bên phải → *Session options*):
- **Accelerator: GPU T4 x2**. Không dùng P100: kiến trúc Pascal, bitsandbytes 4-bit không đảm bảo.
- **Internet: On** (cần xác minh số điện thoại tài khoản Kaggle).
- Chạy nền không cần mở trình duyệt: *Save Version → Save & Run All (Commit)*, tối đa 12 giờ.

**Chạy tiếp khi bị ngắt:** mở phiên mới, *Add Input* → chọn output của version trước. Ô số 2 tự chép
`bench/raw/*.jsonl` cũ sang, `run_infer.py` bỏ qua các lượt đã có.

In [ ]:
# 1. Cài thư viện. Ghim transformers đúng bản đã thử trên máy local (Gemma-4 cần >= 5.5).
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!pip install -q -U "transformers==5.5.0" "bitsandbytes>=0.46" accelerate nvidia-ml-py pyyaml
!python -c "import torch, transformers, bitsandbytes; print('torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bitsandbytes.__version__, '| GPU', torch.cuda.device_count())"

In [ ]:
# 2. Lấy code + bộ thử từ GitHub, chép kết quả cũ nếu có.
import glob, os, shutil, subprocess

REPO, BRANCH = "https://github.com/hoanganhquanCS04/Nico-tick.git", "quan_dev"
WORK = "/kaggle/working/Nico-tick"
if os.path.isdir(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORK], check=True)
os.chdir(WORK)
subprocess.run(["git", "log", "-1", "--format=commit %h %s"], check=True)

os.makedirs("bench/raw", exist_ok=True)
old = [p for p in glob.glob("/kaggle/input/**/bench/raw/*", recursive=True) if p.endswith((".jsonl", ".json"))]
for p in old:
    dst = os.path.join("bench/raw", os.path.basename(p))
    if not os.path.exists(dst) or os.path.getsize(p) > os.path.getsize(dst):
        shutil.copy(p, dst)
print(f"Chép {len(old)} file kết quả cũ" if old else "Chạy mới từ đầu")

# Trọng số mô hình (~19GB) để ở ổ tạm, không để trong /kaggle/working (giới hạn 20GB, bị lưu làm output)
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"

In [ ]:
# 3. Chạy thử: mỗi mô hình 2 prompt x 1 seed, ghi vào bench/smoke (không lẫn với kết quả thật).
#    Mục đích: bắt lỗi nạp mô hình / fp16 tràn số (Gemma trên T4) TRƯỚC khi chạy 4-6 tiếng.
SMOKE = ["qwen35-0.8b", "sailor2-1b", "qwen35-2b", "gemma4-e2b-plecpu"]
for key in SMOKE:
    !CUDA_VISIBLE_DEVICES=0 python bench/run_infer.py --model {key} --seeds 0 --ids F1-001 F2-001 --out bench/smoke
!python bench/score.py --raw bench/smoke --out bench/smoke

In [ ]:
# 3b. Soi output thô của lần chạy thử. Gemma ra chuỗi rỗng / ký tự lặp vô nghĩa = fp16 tràn số
#     -> sửa compute_dtype của gemma trong bench/models.yaml thành float32 rồi chạy lại ô 3.
import json
for p in sorted(glob.glob("bench/smoke/*__seed0.jsonl")):
    for ln in open(p, encoding="utf-8"):
        r = json.loads(ln)
        print(f"== {r['model']} {r['prompt_id']} | {r.get('n_gen_tokens')} token | {r.get('latency_s')}s | VRAM {r.get('vram_nvml_peak_mb')} MB")
        print((r.get("raw_output") or r.get("error") or "")[:400])
        print()

In [ ]:
# 4. Chạy thật: 2 hàng đợi song song, mỗi card một hàng. Chia sao cho thời gian hai bên gần bằng nhau.
#    Log từng mô hình ở bench/logs/. Ô này in tiến độ 5 phút/lần và chờ tới khi cả hai xong.
import subprocess, time, pathlib

QUEUES = {
    0: [("qwen35-2b", []), ("sailor2-1b", [])],
    1: [("gemma4-e2b-plecpu", []), ("qwen35-0.8b", []),
        ("gemma4-e2b-gpu", ["--seeds", "0", "--limit", "10"])],   # dòng tham chiếu VRAM
}
pathlib.Path("bench/logs").mkdir(exist_ok=True)
script = {gpu: " && ".join(f"python -u bench/run_infer.py --model {k} {' '.join(extra)} > bench/logs/{k}.log 2>&1"
                           for k, extra in q) for gpu, q in QUEUES.items()}
procs = {gpu: subprocess.Popen(["bash", "-c", s], env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
         for gpu, s in script.items()}

def tail(path, n=1):
    try:
        return open(path, encoding="utf-8").read().strip().splitlines()[-n:]
    except FileNotFoundError:
        return []

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {(time.time() - t0) / 60:.0f} phút")
    for gpu, q in QUEUES.items():
        for k, _ in q:
            last = tail(f"bench/logs/{k}.log")
            if last:
                print(f"  GPU{gpu} {k}: {last[-1][:160]}")
print({gpu: p.returncode for gpu, p in procs.items()}, f"tổng {(time.time() - t0) / 3600:.1f} giờ")

In [ ]:
# 5. Chấm điểm + bảng kết quả + mẫu chấm tay mù (việc 2.6).
!python bench/score.py --c4-sample

In [ ]:
# 6. Đóng gói để tải về máy (Output -> bench_results.zip). Log thô giữ nguyên, bảng có thể chấm lại ở local.
!cd /kaggle/working/Nico-tick && zip -qr /kaggle/working/bench_results.zip bench/raw bench/logs bench/results.csv bench/table.md bench/formulas.csv eval/c4_blind.csv eval/c4_key.csv
!ls -lh /kaggle/working/bench_results.zip